# Case Study: Event-Synchronized Analysis

This tutorial shows a real-world workflow: matching events from two different logs, applying clock corrections, and generating event-by-event summary plots.

The event windows are still represented by `gwpy.segments.Segment`, while gwexpy provides the table-management and `apply()`-driven batch workflow through `SegmentTable`. The division between GWpy base classes and gwexpy extensions is the same as in [SegmentTable: Basics](../segments/intro_segment_table.ipynb).


In [ ]:
import warnings


import warnings

with warnings.catch_warnings():

    import pandas as pd
    from gwpy.segments import Segment

    from gwexpy.table import SegmentTable

    # 1. Simulate finding events
    events_df = pd.DataFrame({
    "gps": [1234567890.1, 1234567895.5, 1234567900.2],
    "snr": [15, 8, 22]
    })

    # 2. Create SegmentTable from events
    # Define a 4-second window around each GPS time
    segs = [Segment(t-2, t+2) for t in events_df["gps"]]
    st = SegmentTable.from_segments(segs, snr=events_df["snr"])
    st


## Batch Plot Generation

We can use `apply()` to generate plots for each event and save them, storing the paths back in the table.


In [ ]:
import os
import matplotlib.pyplot as plt
from IPython.display import Image, display

os.makedirs("outputs", exist_ok=True)

def generate_event_plot(row):
    path = f"outputs/event_{row.index}.png"
    fig, ax = plt.subplots(figsize=(6, 2.5))
    seg = row["span"]
    snr = row["snr"]
    center = (seg[0] + seg[1]) / 2
    ax.axvspan(
        seg[0], seg[1], color="tab:blue", alpha=0.3,
        label=f"Window [{seg[0]:.1f}, {seg[1]:.1f})",
    )
    ax.axvline(center, color="red", linestyle="--", label=f"Event (SNR={snr})")
    ax.set(
        xlabel="GPS Time [s]", ylabel="Normalized",
        title=f"Event {row.index} Window", xlim=(seg[0] - 1, seg[1] + 1),
    )
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    return {"plot_path": path}

st_plots = st.apply(generate_event_plot)
st_plots.display()

display(Image("outputs/event_0.png"))


## Result Summary

Finally, we can filter the table and export results.


In [ ]:
st_best = st_plots.select(mask=st_plots.to_pandas()["snr"] > 10)
display(st_best.to_pandas())

fig = st_best.hist(column="snr")
